# Informações do Projeto

## TEMA CENTRAL

> *Dada uma determinada janela de horário em um dia específico, é possível prever/classificar o nível de risco de ocorrências criminais (Alto/Baixo) no entorno do campus do IFSP-SPO?*

## Autores

| NOME | PRONTUÁRIO | GITHUB |
| ---- | ---------- | ------ |
| Andrey Gustavo Segantin dos Santos | SP3155439 | [AndreyGustavo267](https://github.com/AndreyGustavo267) |
| Elton Lima Araujo | SP3155544 | [Elton1466](https://github.com/Elton1466) |
| Felipe Teixeira de Lima | SP3155048 | [felipeteixeiradelima](https://github.com/felipeteixeiradelima) |
| João Gustavo dos Santos | SP3154211  | [joaogust](https://github.com/joaogust) |

# 0. Bibliotecas e parâmetros

## 0.1. Importação de bibliotecas

### Bibliotecas de uso geral

In [ ]:
import sys

import folium
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from folium.plugins import HeatMap
from matplotlib import dates as mdates

### Biblioteca _first-party_

In [ ]:
sys.path.append("..")

from modules.data import make_dataset

## 0.2. Parâmetros

### Parâmetros gerais

### Parâmetros de extração do dataset

In [ ]:
# Parâmetros de download
PROXIES: dict | None = None
TIMEOUT_SEGUNDOS: float | tuple | None = None

# Parâmetros de salvamento
DIR_DADOS_BRUTOS: str = "../data/raw/"
DIR_DADOS_PROCESSADOS: str = "../data/processed"
NOME_ARQUIVO_DATASET_CONSOLIDADO: str = "SPDadosCriminais_Unificado.csv"
COLUNAS_DESEJADAS: list[str] = [
    "NUM_BO",
    "BAIRRO",
    "LATITUDE",
    "LONGITUDE",
    "DATA_COMUNICACAO_BO",
    "DATA_OCORRENCIA_BO",
    "HORA_OCORRENCIA_BO",
    "DESCR_PERIODO",
    "RUBRICA",
]

# 1. Extração dos dados

## 1.1. Baixando e consolidando dataset

In [ ]:
make_dataset.make_dataset(
    raw_data_path=DIR_DADOS_BRUTOS,
    processed_data_path=DIR_DADOS_PROCESSADOS,
    dataset_filename=NOME_ARQUIVO_DATASET_CONSOLIDADO,
    proxies=PROXIES,
    timeout=TIMEOUT_SEGUNDOS,
)

## 1.2. Carregando dataset em um DataFrame do Pandas

In [ ]:
df_crimes = pd.read_csv(
    f"{DIR_DADOS_PROCESSADOS}/{NOME_ARQUIVO_DATASET_CONSOLIDADO}",
    sep=";",
    usecols=COLUNAS_DESEJADAS,
)

In [ ]:
df_crimes.head()

# 2. Análise Exploratória de Dados (AED)

## 2.1. Inspeção de estrutura

In [ ]:
# Sumário conciso do DataFrame

df_crimes.info()

In [ ]:
# Estatísticas descritivas do DataFrame

df_crimes.describe()

In [ ]:
# Saúde das colunas do DataFrame

s_nulos = df_crimes.isna().sum()
s_porcentagens = ((s_nulos / len(df_crimes)) * 100).round(2)

df_saude_colunas = pd.DataFrame({"Nulos": s_nulos, "Porcentagem (%)": s_porcentagens})

df_saude_colunas

In [ ]:
# Saúde da coluna NUM_BO (chave)

qtd_nulos = df_crimes["NUM_BO"].isna().sum()
qtd_duplicados = df_crimes["NUM_BO"].duplicated().sum()

print(f"Quantidade de registros com chave 'NUM_BO' nulos: {qtd_nulos} de {len(df_crimes)} ({qtd_nulos/len(df_crimes)*100:.2f}%)")
print(f"Quantidade de registros com chave 'NUM_BO' duplicados: {qtd_duplicados} de {len(df_crimes)} ({qtd_duplicados/len(df_crimes)*100:.2f}%)")

In [ ]:
# Saúde das colunas LATITUDE e LONGITUDE

qtd_valores_nulos_latitude = (df_crimes["LATITUDE"].isna() | df_crimes["LATITUDE"] == 0).sum()
qtd_valores_nulos_longitude = (df_crimes["LONGITUDE"].isna() | df_crimes["LONGITUDE"] == 0).sum()

print(f"Quantidade de valores nulos/inválidos na coluna 'LATITUDE': {qtd_valores_nulos_latitude} de {len(df_crimes)} registros ({(qtd_valores_nulos_latitude / len(df_crimes)) * 100:.2f}%)")
print(f"Quantidade de valores nulos/inválidos na coluna 'LONGITUDE': {qtd_valores_nulos_longitude} de {len(df_crimes)} registros ({(qtd_valores_nulos_longitude / len(df_crimes)) * 100:.2f}%)")

### 2.2. Distribuição temporal e sazonalidade

#### Histograma de data de ocorrência do BO

In [ ]:
sns.set_theme(style="whitegrid")

datas = pd.to_datetime(df_crimes["DATA_OCORRENCIA_BO"], errors="coerce").dropna()

fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(
    datas,
    bins=50,
    color="#4C72B0",
    edgecolor="white",
    linewidth=0.6,
    ax=ax,
)

ax.set_title("Distribuição das ocorrências por data", fontsize=14, weight="bold")
ax.set_xlabel("Data da ocorrência", fontsize=11)
ax.set_ylabel("Quantidade de registros", fontsize=11)

ax.tick_params(axis="x", rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))

fig.tight_layout()
plt.show()

#### Análise da hora de ocorrência do BO

In [ ]:
# Saúde da coluna HORA_OCORRENCIA_BO

qtd_horas_ocorrencia_nulas = df_crimes["HORA_OCORRENCIA_BO"].isna().sum()

print(f"Quantidade de valores nulos na coluna 'HORA_OCORRENCIA_BO': {qtd_horas_ocorrencia_nulas} de {len(df_crimes)} registros ({(qtd_horas_ocorrencia_nulas / len(df_crimes)) * 100:.2f}%)")

In [ ]:
sns.set_theme(style="whitegrid")

horas = pd.to_datetime(
    df_crimes["HORA_OCORRENCIA_BO"].astype(str),
    format="%H:%M:%S",
    errors="coerce",
).dropna()

fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(
    horas,
    bins=50,
    color="#4C72B0",
    edgecolor="white",
    linewidth=0.6,
    ax=ax,
)

ax.set_title("Distribuição das ocorrências por hora", fontsize=14, weight="bold")
ax.set_xlabel("Hora da ocorrência", fontsize=11)
ax.set_ylabel("Quantidade de registros", fontsize=11)

ax.tick_params(axis="x", rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

fig.tight_layout()
plt.show()

#### Checando saúde da coluna de descrição do período

In [ ]:
# Saúde da coluna DESCR_PERIODO

qtd_descricoes_nulas = df_crimes["DESCR_PERIODO"].isna()

print(f"Quantidade de valores nulos na coluna 'DESCR_PERIODO': {qtd_descricoes_nulas.sum()} de {len(df_crimes)} registros ({(qtd_descricoes_nulas.sum() / len(df_crimes)) * 100:.2f}%)")

In [ ]:
categorias_descr_periodo = df_crimes["DESCR_PERIODO"].value_counts().head().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=categorias_descr_periodo.values,
    y=categorias_descr_periodo.index,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="x", style="plain", useOffset=False, useLocale=True)

ax.set_title("Descrições de período mais frequentes", fontsize=14, weight="bold")
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Descrição do período")

fig.tight_layout()
plt.show()

### 2.3. Mapeamento de frequências

##### Bairros

In [ ]:
# Saúde da coluna BAIRRO

qtd_bairros_nulos = df_crimes["BAIRRO"].isna().sum()

print(f"Quantidade de valores nulos na coluna 'BAIRRO': {qtd_bairros_nulos} de {len(df_crimes)} registros ({(qtd_bairros_nulos / len(df_crimes)) * 100:.2f}%)")

In [ ]:
top_bairros = df_crimes["BAIRRO"].value_counts().head(20).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=top_bairros.values,
    y=top_bairros.index,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="x", style="plain", useOffset=False, useLocale=True)

ax.set_title("Top 20 bairros com mais ocorrências", fontsize=14, weight="bold")
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Bairro")

fig.tight_layout()
plt.show()

##### Rubricas

In [ ]:
top_rubricas = df_crimes["RUBRICA"].value_counts().head(10).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=top_rubricas.values,
    y=top_rubricas.index,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="x", style="plain", useOffset=False, useLocale=True)

ax.set_title("Top 10 rubricas criminais mais frequentes", fontsize=14, weight="bold")
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Rubrica")

fig.tight_layout()
plt.show()

### 2.4. Análises bivariadas

##### Tabela de contingência Período x Rúbrica

In [ ]:
pd.crosstab(
    index=df_crimes["DESCR_PERIODO"],
    columns=df_crimes["RUBRICA"],
    margins=True,
    margins_name="Total",
)

In [ ]:
sns.set_theme(style="whitegrid")

dias_da_semana = (
    pd.to_datetime(df_crimes["DATA_OCORRENCIA_BO"], errors="coerce").dropna().dt.weekday
).value_counts().sort_index()

dias_da_semana.index = dias_da_semana.index.map(
    {
        0: "Segunda-feira",
        1: "Terça-feira",
        2: "Quarta-feira",
        3: "Quinta-feira",
        4: "Sexta-feira",
        5: "Sábado",
        6: "Domingo",
    }
)

fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    x=dias_da_semana.index,
    y=dias_da_semana.values,
    color="#4C72B0",
    ax=ax,
)

ax.ticklabel_format(axis="y", style="plain", useOffset=False, useLocale=True)

ax.set_title(
    "Distribuição das ocorrências por dia da semana", fontsize=14, weight="bold"
)
ax.set_xlabel("Quantidade de ocorrências")
ax.set_ylabel("Dia da semana")

fig.tight_layout()
plt.show()

### 2.5. Especial: mapa de calor de ocorrências

In [ ]:
mapa_calor = folium.Map(
    location=[-23.5505, -46.6333],
    zoom_start=11,
    tiles="CartoDB positron",
)

coordenadas = df_crimes[["LATITUDE", "LONGITUDE"]].dropna()

for col in coordenadas.columns:
    coordenadas[col] = pd.to_numeric(coordenadas[col], errors="coerce")

coordenadas = coordenadas.dropna().values.tolist()

HeatMap(coordenadas, radius=15, blur=10).add_to(mapa_calor)

mapa_calor

## Tratando tipos

In [ ]:
df_crimes_tratado = df_crimes.copy()

df_crimes_tratado["DATA_OCORRENCIA_BO"] = pd.to_datetime(df_crimes_tratado["DATA_OCORRENCIA_BO"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
df_crimes_tratado["HORA_OCORRENCIA_BO"] = pd.to_datetime(df_crimes_tratado["HORA_OCORRENCIA_BO"], format="%H:%M:%S", errors="coerce").dt.time
df_crimes_tratado["DESCR_PERIODO"] = df_crimes_tratado["DESCR_PERIODO"].str.strip().str.upper()
df_crimes_tratado["BAIRRO"] = df_crimes_tratado["BAIRRO"].str.strip().str.upper()
df_crimes_tratado["LATITUDE"] = pd.to_numeric(df_crimes_tratado["LATITUDE"], errors="coerce")
df_crimes_tratado["LONGITUDE"] = pd.to_numeric(df_crimes_tratado["LONGITUDE"], errors="coerce")

df_crimes_tratado.head()